<a href="https://colab.research.google.com/github/animesh-kishore/3_chatbots_chatting/blob/main/Copy_of_llama3_2_1b_instruct_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True) # login to huggingface

In [ ]:
from transformers import AutoTokenizer

model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Standard step for tokenizers which don't have a separate PAD token. eos is End of Sequence

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are a helpful assistant'},
    {'role': 'user', 'content': 'Tell me a joke about LLM engineers'},
]

tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) # visualize final tokens

In [ ]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to('cuda') # Tokenize (default true) and return pytorch tensors
inputs

In [ ]:
!pip install -U bitsandbytes>=0.46.1 # update bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# load_in_4bit: Load model weights in 4bits fp4 or nf4
# bnb_4bit_compute_dtype: Actual dtype in which computation happens. 4bit weights are dequantized into torch.bfloat16 before computation.
quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto', quantization_config=quant_cfg) # device_map auto best utilizes underlying HW e.g. GPU, CPU etc

In [ ]:
model

In [ ]:
from transformers import TextStreamer

streamer = TextStreamer(tokenizer) # Stream model output generation

model.generate(**inputs, max_new_tokens=80, streamer=streamer)